In [1]:
from datetime import datetime, timedelta
latest_date = datetime.now()
earliest_date = (latest_date - timedelta(days=64)).strftime(r"%Y-%m-%d")
current_date = latest_date.strftime(r"%Y-%m-%d")

In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 47.52271,
	"longitude": 7.64511,
	"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "precipitation_probability", "weather_code", "wind_speed_10m", "cloud_cover", "surface_pressure", "dew_point_2m", "pressure_msl"],
	"timezone": "Europe/Berlin",
	"start_date": earliest_date,
	"end_date": current_date,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(3).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(6).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(7).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(8).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(9).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["weather_code"] = hourly_weather_code
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["pressure_msl"] = hourly_pressure_msl

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)
print(hourly_dataframe.info())

Coordinates: 47.52000045776367°N 7.639999866485596°E
Elevation: 293.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Hourly data
                           date  temperature_2m  relative_humidity_2m  \
0    2026-02-19 00:00:00+00:00        3.236000                  98.0   
1    2026-02-19 01:00:00+00:00        2.986000                  99.0   
2    2026-02-19 02:00:00+00:00        3.186000                  98.0   
3    2026-02-19 03:00:00+00:00        3.336000                  97.0   
4    2026-02-19 04:00:00+00:00        3.636000                  97.0   
...                        ...             ...                   ...   
1555 2026-04-24 19:00:00+00:00       20.806499                  27.0   
1556 2026-04-24 20:00:00+00:00       19.056499                  31.0   
1557 2026-04-24 21:00:00+00:00       16.456501                  38.0   
1558 2026-04-24 22:00:00+00:00       12.556500                  51.0   
1559 2026-04-24 23:00:00+00:00       11.106501 

In [3]:
###
# Prepare Hopsworks project

import hopsworks
from dotenv import load_dotenv
import os

env = load_dotenv(".env")

project = hopsworks.login(
    project='mlops',  # Replace with your project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.environ["HOPSWORKS_API_KEY"]  # Get from Hopsworks UI > Account Settings > API Keys
)

fs = project.get_feature_store()


2026-04-24 14:57:09,034 INFO: Initializing external client
2026-04-24 14:57:09,035 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-04-24 14:57:11,188 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31926


In [4]:
# Feature Preparation and cleanup

feature_dataframe = hourly_dataframe.copy()

# Add the maximum precipation of the following two rows as a label.
# This will not be able to create a label for the last two rows but should generate enough data for training purposes.
next_row_1 = feature_dataframe['precipitation'].shift(-1)
next_row_2 = feature_dataframe['precipitation'].shift(-2)
# Calculate the maximum of the next two rows
feature_dataframe['precipation_label'] = pd.concat([next_row_1, next_row_2], axis=1).max(axis=1)
# Remove the last two rows
feature_dataframe = feature_dataframe.iloc[:-2]

# Weather Variables could depend on the time of day
feature_dataframe["derived_hour"] = feature_dataframe["date"].apply(lambda x: x.hour)
# Weather Variables could depend on the month of the year
feature_dataframe["derived_month"] = feature_dataframe["date"].apply(lambda x: x.month)
# Generate Partion Key
feature_dataframe["partition"] = feature_dataframe["date"].apply(lambda x: f'{x.year}-{x.month}')
# Rain Probability could depend on the average humidity of the last 24 hours
feature_dataframe["timestamp"] = feature_dataframe["date"] # Copy to save column
feature_dataframe.set_index('date', inplace=True) # Needed for averaging over time later
feature_dataframe['relative_humidity_2m_24h_avg'] = feature_dataframe['relative_humidity_2m'].rolling('24h').mean()
# Rain Probability could depend on if the barometric pressure is rising or falling
feature_dataframe['pressure_msl_3h_pct_change'] = feature_dataframe['pressure_msl'].pct_change(periods=3)

# Find first valid index (during preparation pressure_msl_3h_pct_change returned the latest valid row)
first_non_nan_index = feature_dataframe['pressure_msl_3h_pct_change'].first_valid_index()

# Slice the DataFrame to keep only rows from the first non-NaN index onward
feature_dataframe = feature_dataframe.loc[first_non_nan_index:]


print(feature_dataframe.info())
print(feature_dataframe.head())
print("min_timesamp", feature_dataframe["timestamp"].min())
print("max_timestamp", feature_dataframe["timestamp"].max())



<class 'pandas.DataFrame'>
DatetimeIndex: 1555 entries, 2026-02-19 03:00:00+00:00 to 2026-04-24 21:00:00+00:00
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype             
---  ------                        --------------  -----             
 0   temperature_2m                1555 non-null   float32           
 1   relative_humidity_2m          1555 non-null   float32           
 2   precipitation                 1555 non-null   float32           
 3   precipitation_probability     1555 non-null   float32           
 4   weather_code                  1555 non-null   float32           
 5   wind_speed_10m                1555 non-null   float32           
 6   cloud_cover                   1555 non-null   float32           
 7   surface_pressure              1555 non-null   float32           
 8   dew_point_2m                  1555 non-null   float32           
 9   pressure_msl                  1555 non-null   float32           
 10  precipation

In [5]:
# Initial Creation of FeatureGroup. Only to be run on first run
# fg = fs.create_feature_group("weather_prediction", version=1, primary_key=["timestamp"], event_time="timestamp", partition_key=["partition"], online_enabled=True)
# fg.save(feature_dataframe)

In [6]:
# Concatenate with previous DataFrame (if available)
## Get most current previous feature group as basis to concatenate
## In Retrospect it would probably also have been possible to use the fg.append_features() method. 
## Then it would be unclear if the data values of past times would have been the same.

feature_group_name = "weather_prediction"

try:
    # Get all versions of the feature group
    fgs = fs.get_feature_groups()
    fg_versions = [fg for fg in fgs if fg.name == feature_group_name]

    if not fg_versions:
        print(f"Feature group '{feature_group_name}' does not exist.")
    else:
        # Get the latest version
        latest_version = max(fg_versions, key=lambda x: x.version).version
        fg = fs.get_feature_group(feature_group_name, version=latest_version)

        # Read the data into a pandas DataFrame
        old_feature_dataframe = fg.read()
        print(f"Data loaded successfully for version {latest_version}.")
        print(old_feature_dataframe.info())
except Exception as e:
    print(f"An error occurred: {e}")






Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.53s) 
Data loaded successfully for version 4.
<class 'pandas.DataFrame'>
RangeIndex: 1555 entries, 0 to 1554
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   partition                     1555 non-null   str           
 1   temperature_2m                1555 non-null   float32       
 2   relative_humidity_2m          1555 non-null   float32       
 3   precipitation                 1555 non-null   float32       
 4   precipitation_probability     1555 non-null   float32       
 5   weather_code                  1555 non-null   float32       
 6   wind_speed_10m                1555 non-null   float32       
 7   cloud_cover                   1555 non-null   float32       
 8   surface_pressure              1555 non-null   float32       
 9   dew_point_2m                  1555 non-null   f

In [7]:
## Add new rows (timestamp > max existing timestamp)
## Does not handle switch between Daylight Savings Time but these are only few samples and 
## can be neglected since model Quality is not relevant for project
old_feature_dataframe['timestamp'] = old_feature_dataframe['timestamp'].dt.tz_localize('Europe/Berlin', nonexistent="shift_forward")
latest_timestamp = old_feature_dataframe['timestamp'].max()
new_rows = feature_dataframe[feature_dataframe['timestamp'] > latest_timestamp]
new_feature_dataframe: pd.DataFrame = pd.concat([old_feature_dataframe, new_rows], ignore_index=True).sort_values('timestamp')
new_feature_dataframe.sort_values("timestamp")


,partition,temperature_2m,relative_humidity_2m,precipitation,precipitation_probability,weather_code,wind_speed_10m,cloud_cover,surface_pressure,dew_point_2m,pressure_msl,precipation_label,derived_hour,derived_month,timestamp,relative_humidity_2m_24h_avg,pressure_msl_3h_pct_change
0,2026-2,3.336000,97.0,0.0,31.0,45.0,5.001280,100.0,966.493164,2.905129,1002.000000,1.3,3,2,2026-02-19 03:00:00+01:00,98.000000,-0.001594
1,2026-2,3.636000,97.0,1.3,73.0,61.0,8.891343,100.0,966.723999,3.204080,1002.200012,1.0,4,2,2026-02-19 04:00:00+01:00,97.800000,0.000299
2,2026-2,3.836000,96.0,1.0,100.0,55.0,7.636753,100.0,966.749023,3.256539,1002.200012,0.2,5,2,2026-02-19 05:00:00+01:00,97.500000,-0.000399
3,2026-2,3.886000,96.0,0.2,98.0,51.0,8.209263,100.0,966.562378,3.306304,1002.000000,0.0,6,2,2026-02-19 06:00:00+01:00,97.285714,0.000000
4,2026-2,3.986000,96.0,0.0,81.0,3.0,7.704336,100.0,966.478516,3.405835,1001.900024,0.0,7,2,2026-02-19 07:00:00+01:00,97.125000,-0.000299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1552,2026-4,20.806499,27.0,0.0,0.0,0.0,6.989935,0.0,983.834045,1.115775,1017.799988,0.0,19,4,2026-04-24 19:00:00+02:00,50.458333,-0.001276
1553,2026-4,19.056499,31.0,0.0,0.0,0.0,5.001280,0.0,983.827942,1.530485,1018.000000,0.0,20,4,2026-04-24 20:00:00+02:00,50.375000,-0.000491
1554,2026-4,16.456501,38.0,0.0,0.0,0.0,4.452954,0.0,983.817139,2.088165,1018.299988,0.0,21,4,2026-04-24 21:00:00+02:00,50.291667,0.000295
1555,2026-4,19.056499,31.0,0.0,0.0,0.0,5.001280,0.0,983.827942,1.530485,1018.000000,0.0,20,4,2026-04-24 20:00:00+00:00,50.375000,-0.000491


In [8]:
# Plausibility Checks before Upload
# Due to selected datasource and data preparation, this can only be updated once per day
print(new_feature_dataframe["timestamp"].min())
print(new_feature_dataframe["timestamp"].max())

if new_feature_dataframe["timestamp"].min() != old_feature_dataframe["timestamp"].min():
    raise ValueError("New Feature Dataframe should have the same start timestamp as the Old Feature Dataframe")

if new_feature_dataframe["timestamp"].max() > old_feature_dataframe["timestamp"].max():
    # Upload updated feature dataframe to hopsworks
    new_version = latest_version + 1 # TODO Think if the latest date would be better

    new_fg = fs.create_feature_group("weather_prediction", version=new_version, primary_key=["timestamp"], event_time="timestamp", partition_key=["partition"], online_enabled=True, tags=["champion"])
    try:
        from hopsworks import RestAPIError
        new_fg.save(feature_dataframe)
    except RestAPIError as e:
        print(f"Probably this section was already run if the version already exists\n{e}")
    
    print(f"Feature group {new_fg.name} created with version {new_fg.version}")
else:
    print("No new feature group version needed no new data or ")


2026-02-19 03:00:00+01:00
2026-04-24 21:00:00+00:00
Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/31926/fs/20610/fg/38499


Uploading Dataframe: 100.00% |██████████| Rows 1555/1555 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: weather_prediction_5_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/31926/jobs/named/weather_prediction_5_offline_fg_materialization/executions
Feature group weather_prediction created with version 5
